In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# patten-server GPU: use PyTorch cu124 (matches driver CUDA 12.x)
#   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import (
    train_tabddpm,
    train_forestdiffusion,
    resolve_experiment_device,
    print_experiment_runtime,
)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
secondary_mushroom = fetch_ucirepo(id=848)

# data (as pandas dataframes)
X = secondary_mushroom.data.features
y = secondary_mushroom.data.targets

# metadata
print(secondary_mushroom.metadata)

# variable information
print(secondary_mushroom.variables)

mushroom_data = pd.concat([X, y], axis=1)

target_col = "class"

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Fast dev mode: fewer epochs for generators only (set False for full paper run)
FAST_MODE = True
RUN_QUALITY_EVAL = True
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

# Diffusion generators only
GENERATORS_TO_EVAL = ["ForestDiffusion", "TabDDPM"]

DEVICE = "auto"
EXPERIMENT_DEVICE = resolve_experiment_device(DEVICE)

# Stratified subsample (full dataset has 61069 rows)
_, mushroom_data = train_test_split(
    mushroom_data,
    train_size=N_SAMPLES,
    stratify=mushroom_data[target_col],
    random_state=SEED,
)
mushroom_data = mushroom_data.reset_index(drop=True)

# Handle missing values
numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = [col for col in CATEGORICAL_COLS if col in mushroom_data.columns]

for col in numeric_cols:
    mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
    mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())

for col in categorical_cols:
    fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
    mushroom_data[col] = mushroom_data[col].fillna(fill)

# Label-encode categorical columns (ordinal encoding)
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
    label_encoders[col] = le

# Encoded positive class for binary metrics (poisonous = "p")
pos_label = int(label_encoders[target_col].transform(["p"])[0])

print(f"Encoded {len(categorical_cols)} categorical columns.")
print(f"{target_col} mapping: {dict(zip(label_encoders[target_col].classes_, range(len(label_encoders[target_col].classes_))))}")
print(f"pos_label (encoded poisonous class): {pos_label}")

X = mushroom_data.drop(columns=[target_col])
y = mushroom_data[target_col]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Benchmark subsample + leak-safe split (generators fit train only)
_strat = mushroom_data[target_col] if mushroom_data[target_col].nunique() <= 30 else None
train_real, test_real = train_test_split(
    mushroom_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=_strat,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)

SYNTHETIC_N = N_SAMPLES
DIFFUSION_SEED = SEED
_categorical_columns = [target_col]

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}
synthetic_datasets = {}
synthetic_outputs = {}
quality_results = []

print(f"Subsample: {mushroom_data.shape} | train_real: {train_real.shape} | test_real: {test_real.shape}")
print_experiment_runtime(EXPERIMENT_DEVICE)


{'uci_id': 848, 'name': 'Secondary Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/848/data.csv', 'abstract': 'Dataset of simulated mushrooms for binary classification into edible and poisonous.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 61068, 'num_features': 20, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2021, 'last_updated': 'Wed Apr 10 2024', 'dataset_doi': '10.24432/C5FP5Q', 'creators': ['Dennis Wagner', 'D. Heider', 'Georges Hattab'], 'intro_paper': {'ID': 259, 'type': 'NATIVE', 'title': 'Mushroom data creation, curation, and simulation to support classification tasks', 'authors': 'Dennis Wagner, D. Heider, Georges Hattab', 'venue': 'Scientific Reports', 'year': 2021, 'journal': None, '

In [3]:
# ---------------------------------------------------
# Diffusion models (TabDDPM, ForestDiffusion)
# ---------------------------------------------------
seed = DIFFUSION_SEED

print("\n================ SINGLE RUN ================")
print(f"TabDDPM device: {EXPERIMENT_DEVICE}")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if EXPERIMENT_DEVICE.startswith("cuda"):
    gpu_id = int(EXPERIMENT_DEVICE.split(":")[1]) if ":" in EXPERIMENT_DEVICE else 0
    torch.cuda.set_device(gpu_id)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.empty_cache()

train_diffusion_metadata = SingleTableMetadata()
train_diffusion_metadata.detect_from_dataframe(train_real)

if "TabDDPM" in GENERATORS_TO_EVAL:
    import traceback
    try:
        print("Training TabDDPM...")
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=_categorical_columns,
            n_samples=SYNTHETIC_N,
            seed=seed,
            device=EXPERIMENT_DEVICE,
            fast_mode=FAST_MODE,
        )
        synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_diffusion_metadata,
            )
            scores["TabDDPM"] = quality.get_score()
            print("TabDDPM:", round(scores["TabDDPM"], 4))
        else:
            print("TabDDPM: trained (quality eval skipped)")
    except Exception as e:
        print("TabDDPM Failed:", e)
        traceback.print_exc()
else:
    print("TabDDPM: skipped (not in GENERATORS_TO_EVAL)")

if "ForestDiffusion" in GENERATORS_TO_EVAL:
    import traceback
    try:
        print("Training ForestDiffusion...")
        print(f"ForestDiffusion fast_mode={FAST_MODE} (CPU/XGBoost, n_jobs capped)")
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=_categorical_columns,
            n_samples=SYNTHETIC_N,
            seed=seed,
            fast_mode=FAST_MODE,
        )
        synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()
        print("ForestDiffusion: synthesis complete")
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_diffusion_metadata,
            )
            scores["ForestDiffusion"] = quality.get_score()
            print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))
        else:
            print("ForestDiffusion: trained (quality eval skipped)")
    except Exception as e:
        print("ForestDiffusion Failed (training/sampling):")
        traceback.print_exc()
else:
    print("ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)")

synthetic_outputs = {
    name: synthetic_datasets[name].copy()
    for name in ["TabDDPM", "ForestDiffusion"]
    if name in synthetic_datasets
}
model_order = ["TabDDPM", "ForestDiffusion"]
print(f"model_order: {model_order}")


================ SINGLE RUN ================
TabDDPM device: cuda
Training TabDDPM...
[0]
22
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(22)}
mlp
mlp
Sample timestep    0
Discrete cols: [1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Num shape:  (1000, 20)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 275.08it/s]|
Column Shapes Score: 49.53%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 266.37it/s]|
Column Pair Trends Score: 31.0%

Overall Score (Average): 40.26%

TabDDPM: 0.4026
Training ForestDiffusion...
ForestDiffusion fast_mode=True (CPU/XGBoost, n_jobs capped)
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 113.51it/s]|
Column Shapes Score: 51.74%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 255.03it/s]|
Column Pair Trends

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")

Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [6]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def _normalize_labels(y, reference=None):
    """Align label dtypes so sklearn metrics do not mix strings and numbers."""
    y = pd.Series(y).reset_index(drop=True)
    ref = pd.Series(reference).reset_index(drop=True) if reference is not None else y
    ref_numeric = pd.to_numeric(ref, errors="coerce")
    if ref_numeric.notna().all():
        return pd.to_numeric(y, errors="coerce").round().astype(int)
    return y.astype(str)


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
    use_holdout=False,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    y_test_reference = _normalize_labels(test_df[label_col])

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = _normalize_labels(train_df[label_col], reference=y_test_reference)
            X_test = test_df.drop(columns=[label_col])
            y_test = y_test_reference.copy()

            if use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )

                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_test),
                )

            scaler = StandardScaler().fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)
            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train_s, y_train)
            y_pred = _normalize_labels(clf.predict(X_test_s), reference=y_test_reference)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, average="weighted", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": _std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": _std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": _std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": _std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {_std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {_std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {_std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {_std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col
seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real) — 80% train / 20% holdout")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Diffusion generators: {len(model_order)}"
)
print(f"TRTR training set: {train_real.shape} | Holdout test set: {test_real.shape}")
print(f"Classifier models: {list(models.keys())}")
print(f"Diffusion generators: {model_order}")

trtr_results = evaluate_models(
    train_df=train_real,
    test_df=test_real,
    label_col=label_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds,
    use_holdout=True,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_real,
        label_col=label_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds,
        use_holdout=True,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout
Classifiers: 10 | Seeds: 10 | Diffusion generators: 2
TRTR training set: (48055, 21) | Holdout test set: (12014, 21)
Classifier models: ['LogReg', 'SVM-RBF', 'KNN', 'NaiveBayes', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoost', 'AdaBoost', 'MLP']
Diffusion generators: ['TabDDPM', 'ForestDiffusion']
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
5,RandomForest,0.9998 ± 0.0001,0.9998 ± 0.0001,0.9998 ± 0.0001,0.9998 ± 0.0001
2,KNN,0.9993 ± 0.0002,0.9993 ± 0.0002,0.9993 ± 0.0002,0.9993 ± 0.0002
9,MLP,0.9991 ± 0.0002,0.9991 ± 0.0002,0.9991 ± 0.0002,0.9991 ± 0.0002
4,DecisionTree,0.9425 ± 0.0047,0.9426 ± 0.0047,0.9428 ± 0.0044,0.9425 ± 0.0047
7,GradientBoost,0.8247 ± 0.0039,0.8247 ± 0.0039,0.8247 ± 0.0039,0.8247 ± 0.0039
8,AdaBoost,0.6585 ± 0.0061,0.6594 ± 0.0061,0.6671 ± 0.0056,0.6585 ± 0.0061
0,LogReg,0.6550 ± 0.0007,0.6488 ± 0.0007,0.6530 ± 0.0007,0.6550 ± 0.0007
1,SVM-RBF,0.6541 ± 0.0008,0.6472 ± 0.0009,0.6522 ± 0.0009,0.6541 ± 0.0008
3,NaiveBayes,0.6083 ± 0.0016,0.5949 ± 0.0017,0.6639 ± 0.0023,0.6083 ± 0.0016


TabDDPM - TSTR
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
9,MLP,0.5030 ± 0.0141,0.5012 ± 0.0143,0.5175 ± 0.0157,0.5030 ± 0.0141
1,SVM-RBF,0.4905 ± 0.0083,0.4796 ± 0.0181,0.5116 ± 0.0147,0.4905 ± 0.0083
0,LogReg,0.4901 ± 0.0083,0.4791 ± 0.0180,0.5113 ± 0.0149,0.4901 ± 0.0083
2,KNN,0.4896 ± 0.0193,0.4822 ± 0.0180,0.5105 ± 0.0243,0.4896 ± 0.0193
5,RandomForest,0.4743 ± 0.0250,0.4630 ± 0.0293,0.4943 ± 0.0293,0.4743 ± 0.0250
6,ExtraTrees,0.4740 ± 0.0112,0.4633 ± 0.0076,0.4956 ± 0.0192,0.4740 ± 0.0112
4,DecisionTree,0.4709 ± 0.0326,0.4664 ± 0.0331,0.4834 ± 0.0326,0.4709 ± 0.0326
7,GradientBoost,0.4648 ± 0.0418,0.4499 ± 0.0504,0.4830 ± 0.0440,0.4648 ± 0.0418
8,AdaBoost,0.4568 ± 0.0239,0.3750 ± 0.0679,0.4832 ± 0.0530,0.4568 ± 0.0239
3,NaiveBayes,0.4396 ± 0.0059,0.3158 ± 0.0321,0.4308 ± 0.0406,0.4396 ± 0.0059


TabDDPM - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,0.9117 ± 0.0055,0.9115 ± 0.0055,0.9120 ± 0.0055,0.9117 ± 0.0055
2,KNN,0.8419 ± 0.0060,0.8416 ± 0.0061,0.8417 ± 0.0060,0.8419 ± 0.0060
9,MLP,0.8329 ± 0.0087,0.8316 ± 0.0092,0.8346 ± 0.0079,0.8329 ± 0.0087
5,RandomForest,0.8211 ± 0.0068,0.8210 ± 0.0069,0.8214 ± 0.0078,0.8211 ± 0.0068
4,DecisionTree,0.7629 ± 0.0216,0.7625 ± 0.0225,0.7640 ± 0.0229,0.7629 ± 0.0216
7,GradientBoost,0.7553 ± 0.0123,0.7552 ± 0.0126,0.7557 ± 0.0132,0.7553 ± 0.0123
8,AdaBoost,0.6644 ± 0.0142,0.6605 ± 0.0192,0.6640 ± 0.0141,0.6644 ± 0.0142
0,LogReg,0.6523 ± 0.0032,0.6447 ± 0.0039,0.6506 ± 0.0032,0.6523 ± 0.0032
1,SVM-RBF,0.6490 ± 0.0034,0.6407 ± 0.0043,0.6473 ± 0.0035,0.6490 ± 0.0034
3,NaiveBayes,0.6096 ± 0.0056,0.5970 ± 0.0088,0.6632 ± 0.0062,0.6096 ± 0.0056


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,ForestDiffusion,ExtraTrees,0.088272,0.088467,0.088021,0.088272,1.0000 ± 0.0000,0.9117 ± 0.0055
1,ForestDiffusion,RandomForest,0.178692,0.178739,0.178336,0.178692,0.9998 ± 0.0001,0.8211 ± 0.0068
2,ForestDiffusion,KNN,0.157425,0.157716,0.157571,0.157425,0.9993 ± 0.0002,0.8419 ± 0.0060
3,ForestDiffusion,MLP,0.166140,0.167514,0.164489,0.166140,0.9991 ± 0.0002,0.8329 ± 0.0087
4,ForestDiffusion,DecisionTree,0.179674,0.180055,0.178752,0.179674,0.9425 ± 0.0047,0.7629 ± 0.0216
5,ForestDiffusion,GradientBoost,0.069461,0.069513,0.068981,0.069461,0.8247 ± 0.0039,0.7553 ± 0.0123
6,ForestDiffusion,AdaBoost,-0.005960,-0.001101,0.003129,-0.005960,0.6585 ± 0.0061,0.6644 ± 0.0142
7,ForestDiffusion,LogReg,0.002763,0.004042,0.002455,0.002763,0.6550 ± 0.0007,0.6523 ± 0.0032
8,ForestDiffusion,SVM-RBF,0.005169,0.006537,0.004986,0.005169,0.6541 ± 0.0008,0.6490 ± 0.0034
9,ForestDiffusion,NaiveBayes,-0.001315,-0.002080,0.000689,-0.001315,0.6083 ± 0.0016,0.6096 ± 0.0056


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,0.084032,0.084940,0.084741,0.084032
1,TabDDPM,0.358774,0.384021,0.348073,0.358774


In [9]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")


Results saved to: TRTR_TSTR_results.xlsx
